# Colab 27 — Discrimination figures: "can the encoder separate high-similarity pairs?"

A presentation-scoped notebook for the **AUROC / separation** message, replacing the
true-vs-predicted scatter (which misleads: class imbalance makes one blob, and it invites a
regression `y=x` reading the CE encoder never promises). AUROC asks a **ranking** question — *do
high-sim pairs get higher predicted similarity than low/mid ones?* — so we visualise it that way.

**The message:** *low/mid pairs are not ranked as high.*

Three slide-ready options (same data, same `AdaptiveAvgPool` model as colab24e/19):
- **A — predicted-score box + strip, `high` vs `low/mid`, with the separation threshold line**  *(recommended)*
- **B — overlapping histograms of predicted similarity** (the textbook AUROC companion: AUROC = separation)
- **C — AUROC bar across AA / SS / 3Di** with bootstrap CIs (the summary number)

Per feed the *in-domain-appropriate* encoder is used: AA feed → AA-encoder, SS feed → SS-encoder,
3Di feed → AA-encoder (cross-rep; no 3Di encoder exists — labelled). AUROC positives = pairs with
`current_normlev ≥ 0.70`, negatives = the rest; discriminator = predicted similarity
`1 − ‖e_a − e_b‖ / 2`. **AA high = n_pos 5** (underpowered → read on SS/3Di too).


## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
import os
for f in ['cath_s20_pairs_sample.csv.gz', 'cath_s20_pairs_high.csv.gz',
          'cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz', 'cath_s20_3di.csv.gz']:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch rapidfuzz scikit-learn matplotlib --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve
from rapidfuzz.distance import Levenshtein as RFLevenshtein

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 2. Constants + helpers + architecture (identical to colab24e / colab19)

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'; SS_ALPHABET = 'HLS'
VOCAB_SIZE = len(AA_ALPHABET) + 1; PAD_IDX = len(AA_ALPHABET)
AA_SET = set(AA_ALPHABET); SS_SET = set(SS_ALPHABET)
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}
MIN_LEN, MAX_LEN = 50, 200
N_TRAIN, NUM_EPOCHS, BATCH_SIZE, K = 30000, 30, 128, 16
BAND_LOW_AA, BAND_LOW_SS, BAND_HIGH = 0.30, 0.56, 0.70
BIN_NAMES = ['far', 'mid', 'high']
SEED_TRAIN_AA, SEED_TRAIN_SS = 42, 43
print('recipe: AdaptiveAvgPool K=16, 3-bin CE head, high cutoff', BAND_HIGH)

In [ ]:
def norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - RFLevenshtein.distance(a, b) / L
def is_standard_aa(seq): return all(c in AA_SET for c in seq)
def is_standard_ss(seq): return all(c in SS_SET for c in seq)
def encode_pad(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    idx = [CHAR_TO_IDX[c] for c in seq][:max_len]; idx += [pad_idx] * (max_len - len(idx))
    return torch.tensor(idx, dtype=torch.long)
def perturb(seq, k, alphabet, rng, max_len=MAX_LEN):
    s = list(seq); abc = list(alphabet)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= max_len: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s)); s[i] = rng.choice([c for c in abc if c != s[i]])
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1); s.insert(i, rng.choice(abc))
        else:
            i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)
def random_seq(alphabet, rng, min_len=MIN_LEN, max_len=MAX_LEN):
    return ''.join(rng.choice(list(alphabet), size=int(rng.integers(min_len, max_len + 1))))
def bin_idx_for(x, band_low):
    return 0 if x < band_low else (1 if x < BAND_HIGH else 2)
def make_full_range_pairs(alphabet, n_pairs, rng):
    pairs = []; att = 0
    while len(pairs) < n_pairs and att < n_pairs * 4:
        att += 1; seed = random_seq(alphabet, rng)
        if len(seed) < MIN_LEN: continue
        L = len(seed); t = float(rng.uniform(0, 1)); k = max(0, int(round((1 - t) * L)))
        other = perturb(seed, k, alphabet, rng)
        if 1 <= len(other) <= MAX_LEN: pairs.append((seed, other, norm_lev(seed, other)))
    return pairs

In [ ]:
class SiameseEncoder(nn.Module):
    def __init__(self, K=K, vocab_size=VOCAB_SIZE, embed_dim=32, hidden1=32, hidden2=64,
                 out_dim=128, pad_idx=PAD_IDX):
        super().__init__(); self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, 3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, 3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(K); self.fc = nn.Linear(hidden2 * K, out_dim)
    def forward(self, x):
        mask = (x != self.pad_idx).float()
        e = self.embedding(x).permute(0, 2, 1)
        h = F.relu(self.conv1(e)); h = F.relu(self.conv2(h)); h = h * mask.unsqueeze(1)
        return F.normalize(self.fc(self.pool(h).flatten(1)), p=2, dim=1)

class SiameseClassifier(nn.Module):
    def __init__(self, embed_out=128, hidden_mlp=64, n_bins=3):
        super().__init__(); self.encoder = SiameseEncoder()
        self.head = nn.Sequential(nn.Linear(embed_out, hidden_mlp), nn.LeakyReLU(0.01),
                                  nn.Linear(hidden_mlp, n_bins))
    def forward(self, a, b): return self.head(torch.abs(self.encoder(a) - self.encoder(b)))

class PairDatasetCE(Dataset):
    def __init__(self, pairs, band_low):
        self.a = [encode_pad(a) for a, _, _ in pairs]; self.b = [encode_pad(b) for _, b, _ in pairs]
        self.bins = torch.tensor([bin_idx_for(l, band_low) for _, _, l in pairs], dtype=torch.long)
    def __len__(self): return len(self.bins)
    def __getitem__(self, i): return self.a[i], self.b[i], self.bins[i]

def train_encoder(alphabet, band_low, label, seed):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    pairs = make_full_range_pairs(alphabet, N_TRAIN, rng)
    dl = DataLoader(PairDatasetCE(pairs, band_low), batch_size=BATCH_SIZE, shuffle=True)
    model = SiameseClassifier().to(device); opt = torch.optim.Adam(model.parameters(), 1e-3)
    crit = nn.CrossEntropyLoss(); model.train()
    for ep in range(1, NUM_EPOCHS + 1):
        tot = nb = 0
        for a, b, y in dl:
            a, b, y = a.to(device), b.to(device), y.to(device)
            loss = crit(model(a, b), y); opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        if ep % 10 == 0 or ep == 1: print(f'  [{label}] epoch {ep:3d} CE={tot/nb:.4f}')
    model.eval(); return model

## 3. Pools + feed-matched eval (identical protocol to colab24e)

In [ ]:
raw = pd.concat([pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_train70.csv.gz')),
                 pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_test30.csv.gz'))],
                ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_3di.csv.gz'))
RESCUED = {'4z0mC02', '3qkaE02'}
def _valid(seq, is_std, d):
    return (isinstance(seq, str) and is_std(seq)
            and ((MIN_LEN <= len(seq) <= MAX_LEN) or d in RESCUED))
id_to_aa  = {d: s for d, s in zip(raw['domain_id'], raw['aa_seq']) if _valid(s, is_standard_aa, d)}
id_to_ss  = {d: s for d, s in zip(raw['domain_id'], raw['ss_seq']) if _valid(s, is_standard_ss, d)}
id_to_3di = {d: s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if _valid(s, is_standard_aa, d)}
LOOKUP = {'AA': id_to_aa, 'SS': id_to_ss, '3Di': id_to_3di}
POOL_SET = {f: set(LOOKUP[f]) for f in LOOKUP}
FEEDS = ['AA', 'SS', '3Di']

PAIR_COLS = ['domain_a', 'domain_b', 'ss_score', 'aa_score', 'src4', 'src5', 'src_flag']
samp = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_pairs_sample.csv.gz'), header=None, names=PAIR_COLS)
high = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_pairs_high.csv.gz'), header=None, names=PAIR_COLS)
src = pd.concat([samp, high], ignore_index=True)
src = src[src['domain_a'] != src['domain_b']]
src['pk'] = [frozenset((a, b)) for a, b in zip(src['domain_a'], src['domain_b'])]
src = src.drop_duplicates('pk').reset_index(drop=True)

def build_feed_eval(feed):
    inp = POOL_SET[feed]; lk = LOOKUP[feed]
    sub = src[src['domain_a'].isin(inp) & src['domain_b'].isin(inp)]
    a = sub['domain_a'].values; b = sub['domain_b'].values
    cur = np.array([norm_lev(lk[x], lk[y]) for x, y in zip(a, b)])
    return pd.DataFrame({'domain_a': a, 'domain_b': b, 'current_normlev': cur,
                         'is_high': (cur >= BAND_HIGH).astype(int)})
EVAL = {f: build_feed_eval(f) for f in FEEDS}
for f in FEEDS:
    print(f'{f:<4} eligible={len(EVAL[f]):>6}  high={int(EVAL[f].is_high.sum()):>5}  pool={len(LOOKUP[f]):>6}')

## 4. Train the two encoders + predicted similarity per feed

In [ ]:
model_aa = train_encoder(AA_ALPHABET, BAND_LOW_AA, 'AA', SEED_TRAIN_AA)
model_ss = train_encoder(SS_ALPHABET, BAND_LOW_SS, 'SS', SEED_TRAIN_SS)

# in-domain-appropriate encoder per feed (3Di has no in-domain encoder -> cross-rep via AA)
FEED_ENC = {'AA': ('AA-enc', model_aa), 'SS': ('SS-enc', model_ss), '3Di': ('AA-enc (cross-rep)', model_aa)}

@torch.no_grad()
def pred_sim(model, A, B, batch=512):
    model.eval(); out = []
    for i in range(0, len(A), batch):
        xa = torch.stack([encode_pad(s) for s in A[i:i+batch]]).to(device)
        xb = torch.stack([encode_pad(s) for s in B[i:i+batch]]).to(device)
        d = torch.linalg.vector_norm(model.encoder(xa) - model.encoder(xb), dim=1)
        out.append((1.0 - d / 2.0).cpu().numpy())
    return np.concatenate(out)

PRED = {}   # feed -> dict(pred, y, true, auroc, thr, enc_label, n_pos)
for feed in FEEDS:
    enc_label, model = FEED_ENC[feed]; e = EVAL[feed]; lk = LOOKUP[feed]
    p = pred_sim(model, [lk[d] for d in e['domain_a']], [lk[d] for d in e['domain_b']])
    y = e['is_high'].values.astype(int)
    auroc = float(roc_auc_score(y, p))
    fpr, tpr, thr = roc_curve(y, p); j = int(np.argmax(tpr - fpr))   # Youden-optimal threshold
    PRED[feed] = {'pred': p, 'y': y, 'true': e['current_normlev'].values, 'auroc': auroc,
                  'thr': float(thr[j]), 'enc_label': enc_label, 'n_pos': int(y.sum()),
                  'fpr': fpr, 'tpr': tpr}
    print(f'{feed:<4} [{enc_label:<18}] AUROC={auroc:.3f}  n_pos={int(y.sum()):>4}  '
          f'opt-thr={thr[j]:.3f}')

## 5. Figure A (RECOMMENDED) — predicted-score box + strip: `high` vs `low/mid`

Each feed: predicted similarity for `high (≥0.70)` vs `low/mid (<0.70)` pairs, with the
AUROC-optimal **separation line** (dashed). High sits above the line, low/mid below — *low/mid pairs
are not ranked as high.* Big low/mid group shown as a box + light subsampled strip (no
imbalance-driven blob); all high points shown (red).

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 12})
rng_fig = np.random.default_rng(SEED)
COL_LOWMID, COL_HIGH = '#7f7f7f', '#d62728'

fig, axes = plt.subplots(1, 3, figsize=(15, 5.6), sharey=True)
for ax, feed in zip(axes, FEEDS):
    d = PRED[feed]; p, y = d['pred'], d['y']
    lo, hi = p[y == 0], p[y == 1]
    bp = ax.boxplot([lo, hi], positions=[1, 2], widths=0.55, showfliers=False, patch_artist=True,
                    medianprops=dict(color='black', lw=2))
    for patch, c in zip(bp['boxes'], [COL_LOWMID, COL_HIGH]):
        patch.set_facecolor(c); patch.set_alpha(0.25)
    lo_s = lo if len(lo) <= 600 else lo[rng_fig.choice(len(lo), 600, replace=False)]
    ax.scatter(1 + rng_fig.uniform(-0.16, 0.16, len(lo_s)), lo_s, s=6, color=COL_LOWMID, alpha=0.35, linewidths=0)
    ax.scatter(2 + rng_fig.uniform(-0.16, 0.16, len(hi)), hi, s=55, color=COL_HIGH,
               edgecolors='black', linewidths=0.6, zorder=5)
    ax.axhline(d['thr'], ls='--', color='black', lw=1.4)
    ax.text(2.5, d['thr'], f' separation\n threshold', va='center', ha='left', fontsize=9)
    ax.set_xticks([1, 2]); ax.set_xticklabels([f'low/mid\n(<0.70)\nn={len(lo)}', f'high\n(≥0.70)\nn={len(hi)}'])
    ax.set_title(f'{feed}  ({d["enc_label"]})\nAUROC = {d["auroc"]:.3f}', fontsize=12)
    ax.set_xlim(0.4, 3.1); ax.grid(True, axis='y', alpha=0.3)
axes[0].set_ylabel('predicted similarity\n1 − ‖e_a − e_b‖ / 2')
fig.suptitle('Can the encoder separate high-similarity pairs?  Low/mid pairs are NOT ranked as high.',
             fontsize=14, y=1.02)
plt.tight_layout(); plt.savefig('colab27_figA_separation_box.png', dpi=150, bbox_inches='tight'); plt.show()

## 6. Figure B (companion) — overlapping histograms: AUROC = separation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for ax, feed in zip(axes, FEEDS):
    d = PRED[feed]; p, y = d['pred'], d['y']
    bins = np.linspace(min(p.min(), 0.3), 1.0, 30)
    ax.hist(p[y == 0], bins=bins, color=COL_LOWMID, alpha=0.55, density=True, label=f'low/mid (n={int((y==0).sum())})')
    ax.hist(p[y == 1], bins=bins, color=COL_HIGH, alpha=0.65, density=True, label=f'high (n={int((y==1).sum())})')
    ax.axvline(d['thr'], ls='--', color='black', lw=1.3)
    ax.set_title(f'{feed} — AUROC = {d["auroc"]:.3f}', fontsize=12)
    ax.set_xlabel('predicted similarity'); ax.legend(fontsize=9, framealpha=0.9)
    if ax is axes[0]: ax.set_ylabel('density')
fig.suptitle('Predicted-similarity distributions: little overlap between high and low/mid = high AUROC',
             fontsize=13, y=1.03)
plt.tight_layout(); plt.savefig('colab27_figB_separation_hist.png', dpi=150, bbox_inches='tight'); plt.show()

## 7. Figure C (summary) — AUROC bar across feeds (bootstrap 95% CI)

In [ ]:
def auroc_ci(p, y, n_boot=2000):
    r = np.random.default_rng(SEED); n = len(y); b = []
    for _ in range(n_boot):
        bi = r.integers(0, n, n); yb = y[bi]
        if yb.min() != yb.max(): b.append(roc_auc_score(yb, p[bi]))
    b = np.array(b)
    return (np.percentile(b, 2.5), np.percentile(b, 97.5)) if b.size else (np.nan, np.nan)

aurocs = [PRED[f]['auroc'] for f in FEEDS]
cis = [auroc_ci(PRED[f]['pred'], PRED[f]['y']) for f in FEEDS]
err = np.array([[a - lo for a, (lo, hi) in zip(aurocs, cis)],
                [hi - a for a, (lo, hi) in zip(aurocs, cis)]])
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(FEEDS, aurocs, color=['#1f77b4', '#2ca02c', '#9467bd'], width=0.6)
ax.errorbar(np.arange(len(FEEDS)), aurocs, yerr=err, fmt='none', ecolor='black', capsize=5, lw=1.3)
ax.axhline(0.5, ls='--', color='grey'); ax.text(len(FEEDS)-0.5, 0.515, 'chance (0.5)', ha='right', color='grey')
ax.bar_label(bars, fmt='%.3f', padding=3)
ax.set_ylim(0, 1.06); ax.set_ylabel('AUROC (is-high ≥ 0.70)')
ax.set_title('Separation of high-similarity pairs by feed')
for i, f in enumerate(FEEDS):
    ax.annotate(f'n_pos={PRED[f]["n_pos"]}\n{PRED[f]["enc_label"]}', (i, 0), (0, -30),
                textcoords='offset points', ha='center', va='top', fontsize=8, color='dimgray')
plt.tight_layout(); plt.savefig('colab27_figC_auroc_bar.png', dpi=150, bbox_inches='tight'); plt.show()

## How to read / which to put on the slide

- **Figure A (recommended slide figure).** Two groups per feed; the dashed line is the separation
  threshold. *High pairs sit above it, low/mid below* — your exact message, with the imbalance handled
  (box + subsampled strip, not a blob) and no misleading `y=x` diagonal. AUROC in each title.
- **Figure B.** The textbook AUROC companion — the two predicted-score distributions barely overlap;
  that non-overlap *is* the AUROC. Good if you want to say "AUROC = area of separation" explicitly.
- **Figure C.** The one-number-per-feed summary with CIs (AA wide because n_pos=5).

**Honest caveats (say them):** AUROC is a **pairwise discrimination** measure (high vs the rest) — it
is *not* full-pool retrieval (cf. colab24e MAP@10; the ≥0.70 bar is ill-posed in SS/3Di, where many
pool proteins tie). AA high is **n_pos=5** (read SS/3Di for power). 3Di uses the AA encoder
(cross-rep; no 3Di encoder). No biological claim — relevance is the exact-Levenshtein high-string-
similarity set.